In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR API KEY"

In [48]:
!pip install -qU langchain pinecone-client pypdf langchain-community langchain-google-genai langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [57]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_pinecone import Pinecone

# Initialize the Chat Model (e.g., Gemini 1.5 Pro or Flash)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)

# Initialize Embeddings for Vector Stores
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
import os
from pinecone import Pinecone as PineconeClient # Renamed to avoid conflict with LangChain Pinecone

# TODO: Replace with your actual Pinecone API key
os.environ["PINECONE_API_KEY"] = "YOUR API KEY"

# Initialize Pinecone Client without an explicit environment parameter as per user request
pinecone_client = PineconeClient(api_key=os.environ.get("PINECONE_API_KEY"))

In [61]:
from pinecone import ServerlessSpec # Import ServerlessSpec

# Define Pinecone index name
index_name = "ipl-players"

# Check if index exists, and create if it doesn't
if index_name not in pinecone_client.list_indexes():
    print(f"Creating Pinecone index: {index_name}...")
    pinecone_client.create_index(
        name=index_name,
        dimension=768, # models/gemini-embedding-001 output dimension
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws', # Or 'gcp', 'azure' based on your Pinecone setup
            region='YOUR_PINECONE_REGION', # TODO: Replace with your actual Pinecone region (e.g., 'us-west-2')
        )
    )
    print(f"Pinecone index {index_name} created.")
else:
    print(f"Pinecone index {index_name} already exists.")

# Initialize Pinecone vector store
vector_store = Pinecone(
    index=pinecone_client.Index(index_name),
    embedding=embeddings,
    text_key="text"
)

Creating Pinecone index: ipl-players...


NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '9ae2c9ced962fc4232eaf56e8d462b08', 'date': 'Tue, 25 Aug 2026 19:34:56 GMT', 'server': 'Google Frontend', 'Content-Length': '114', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource cloud: aws region: YOUR_PINECONE_REGION not found"},"status":404}


In [21]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [22]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [35]:
# This cell previously initialized the Chroma vector store, which is no longer needed.
# The Pinecone vector store has been initialized in a preceding cell.

In [36]:
# Add documents to the Pinecone vector store
# The add_documents method returns the IDs of the added documents
pinecone_doc_ids = vector_store.add_documents(docs)
print(f"Added {len(pinecone_doc_ids)} documents to Pinecone with IDs: {pinecone_doc_ids}")

['591bb409-ede5-401d-907a-f759abe8e90e',
 '01b8be6e-4945-4cba-bd39-336b4d40c26c',
 'a2647853-d131-42d2-b007-68e35aefcb4b',
 'bbbd67eb-afc8-43d6-ad3a-12a794288fb6',
 '9132921e-370e-470c-9692-1389e131e769']

In [37]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2bad64cb-51ff-4495-9200-5c9b76e73d60',
  '677aecdc-3925-4b26-8e4f-ec2538c5a608',
  '1b4fa3d3-7ee2-4d58-85b6-d6b98c971223',
  'bf4231c4-6fb5-4368-aa98-90267382aa85',
  '94e81e3f-d115-4442-96a8-227570df64c4',
  '591bb409-ede5-401d-907a-f759abe8e90e',
  '01b8be6e-4945-4cba-bd39-336b4d40c26c',
  'a2647853-d131-42d2-b007-68e35aefcb4b',
  'bbbd67eb-afc8-43d6-ad3a-12a794288fb6',
  '9132921e-370e-470c-9692-1389e131e769'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        ...,
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698, 

In [40]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=3
)

[Document(id='bf4231c4-6fb5-4368-aa98-90267382aa85', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='bbbd67eb-afc8-43d6-ad3a-12a794288fb6', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='9132921e-370e-470c-9692-1389e131e769', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [41]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='bf4231c4-6fb5-4368-aa98-90267382aa85', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(id='bbbd67eb-afc8-43d6-ad3a-12a794288fb6', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053)]

In [43]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="batsman",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='94e81e3f-d115-4442-96a8-227570df64c4', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6921105980873108),
 (Document(id='9132921e-370e-470c-9692-1389e131e769', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6921105980873108),
 (Document(id='1b4fa3d3-7ee2-4d58-85b6-d6b98c971223', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.7129444479942322),
 (Document(id='a2647853-d131-42d2-b007-68e35aefcb4b', m

In [44]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [45]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2bad64cb-51ff-4495-9200-5c9b76e73d60',
  '677aecdc-3925-4b26-8e4f-ec2538c5a608',
  '1b4fa3d3-7ee2-4d58-85b6-d6b98c971223',
  'bf4231c4-6fb5-4368-aa98-90267382aa85',
  '94e81e3f-d115-4442-96a8-227570df64c4',
  '591bb409-ede5-401d-907a-f759abe8e90e',
  '01b8be6e-4945-4cba-bd39-336b4d40c26c',
  'a2647853-d131-42d2-b007-68e35aefcb4b',
  'bbbd67eb-afc8-43d6-ad3a-12a794288fb6',
  '9132921e-370e-470c-9692-1389e131e769'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        ...,
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698, 

In [46]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [47]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2bad64cb-51ff-4495-9200-5c9b76e73d60',
  '677aecdc-3925-4b26-8e4f-ec2538c5a608',
  '1b4fa3d3-7ee2-4d58-85b6-d6b98c971223',
  'bf4231c4-6fb5-4368-aa98-90267382aa85',
  '94e81e3f-d115-4442-96a8-227570df64c4',
  '591bb409-ede5-401d-907a-f759abe8e90e',
  '01b8be6e-4945-4cba-bd39-336b4d40c26c',
  'a2647853-d131-42d2-b007-68e35aefcb4b',
  'bbbd67eb-afc8-43d6-ad3a-12a794288fb6',
  '9132921e-370e-470c-9692-1389e131e769'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        ...,
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698, 